# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and perform basic analysis of the FAIR<sup>2</sup> dataset using the `mlcroissant` library. It follows the recommended steps and references all data entities via their `@id` identifiers.

### Dataset Source
This dataset is described via a Croissant schema available at the following URL:

In [ ]:
# Install mlcroissant if not available
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and examine top-level dataset information using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the Croissant dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Identifier: {metadata.identifier}")
if hasattr(metadata, 'keywords'):
    print(f"Keywords: {', '.join(metadata.keywords)}\n")
print(f"License: {metadata.license}\n")
print(f"Authors: ")
if hasattr(metadata, 'author'):
    for author in metadata.author:
        print(f"- {author['@id']}")
print(f"Publication Date: {metadata.datePublished}")

## 2. Data Overview
Let us examine the available record sets and the fields within each record set. All `@id`s are referenced explicitly. 

By iterating over all discovered record sets, we summarize their structure using their `@id`, and then for each, we list their fields and columns (again by their `@id`).

In [ ]:
# List all available record sets by @id
record_set_ids = dataset.record_set_ids
print(f"Found {len(record_set_ids)} record set(s):")
for i, rs_id in enumerate(record_set_ids):
    print(f"{i+1}. Record Set @id: {rs_id}")
    rec_set = dataset.record_set(rs_id)

    # List fields (by @id)
    if hasattr(rec_set, 'fields'):
        print(f"   Fields:")
        for field in rec_set.fields:
            print(f"     - {field['@id']}")
    # List columns (by @id)
    if hasattr(rec_set, 'columns') and rec_set.columns:
        print(f"   Columns:")
        for column in rec_set.columns:
            print(f"     - {column['@id']}")
    print('')

## 3. Data Extraction
We will load the records for **each record set** using the `@id`. Results are stored in a dictionary of pandas DataFrames using the record set `@id` as key. Columns reflect the field and column `@id`s, in line with the template.

Please refer to the previous cell's output for the record set(s) and field `@id`s.

In [ ]:
# Extract all data into DataFrames by record set @id
dataframes = {}
all_columns = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    all_columns[rs_id] = df.columns.tolist()
    print(f"Record set @id: {rs_id}")
    print(f"Columns (@id): {df.columns.tolist()}\n")
# For demonstration, display head of first record set
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"First 5 rows for record set {first_rs_id}:")
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
For analysis, select a numeric field (by its `@id`) from a record set of interest. Below, we filter by a threshold, normalize the field, and group the data if a suitable categorical field (`@id`) is present.

*Be sure to reference fields by their `@id` as shown above.*

In [ ]:
# Example EDA: Numeric field analysis on first record set (customize as appropriate)
if record_set_ids:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]

    # Attempt to choose a numeric field automatically from the columns
    numeric_field_id = None
    for col in df.select_dtypes(include='number').columns:
        numeric_field_id = col
        break
    # Fall back to the first column if none detected
    if numeric_field_id is None and len(df.columns) > 0:
        numeric_field_id = df.columns[0]

    print(f"Using numeric field @id: {numeric_field_id}")
    # Set an example threshold for filtering (use the median/mean as a guide)
    if numeric_field_id in df.columns:
        try:
            threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
            filtered_df = df[df[numeric_field_id] > threshold]
            print(f"Filtered records with {numeric_field_id} > {threshold:.2f}: {len(filtered_df)} records\n")

            field_norm = f"{numeric_field_id}_normalized"
            filtered_df[field_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
            print(f"Normalized {numeric_field_id} for filtered records (first 5 rows):")
            display(filtered_df[[numeric_field_id, field_norm]].head())

            # Try to group by the first non-numeric column
            group_field = None
            for col in df.columns:
                if col != numeric_field_id and not pd.api.types.is_numeric_dtype(df[col]):
                    group_field = col
                    break
            if group_field:
                print(f"\nGrouped by field @id: {group_field}")
                grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
                display(grouped_df.head())
        except Exception as e:
            print("No numeric data available or error during EDA:", e)
    else:
        print(f"No suitable numeric columns found in record set {record_set_id}.")

## 5. Visualization
We will visualize the distribution of a selected numeric field and, if available, display its relation to a categorical/grouping field (by `@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Reuse EDA selection
if record_set_ids and numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20, color='skyblue')
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If grouping field exists, show boxplot by group
    if group_field and group_field in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(data=df, x=group_field, y=numeric_field_id)
        plt.title(f'{numeric_field_id} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion

This notebook guided you through loading a structured dataset via its Croissant schema, exploring its structure by `@id`, extracting data into DataFrames, and conducting initial statistical and visual analyses. We demonstrated how to reference all record sets, fields, and columns using their `@id` for seamless, standards-based interoperability with other FAIR datasets.

Further exploration could include detailed model interpretability, missing value handling, or advanced visualizations tailored to the domain-specific context of rangeland management and knowledge adoption predictors.